# Эксперименты с разными методами

Ноутбук содержит разведочный анализ данных, сравнение методов
текстового поиска и подготовку финального решения для ранжирования
НПА по описанию товара в таможенной декларации.

Основные этапы:
- анализ и предварительная обработка данных;
- BM25 как лексический baseline;
- семантический поиск с BGE-M3;
- диагностическое сравнение методов;
- формирование итогового top-10.

Эксперименты выполнялись в Google Colab на GPU.

## Импорт библиотек и загрузка данных

Подготовка окружения

In [ ]:
!python project_setup.py

Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import torch

from pathlib import Path

from src.data import (
    load_declarations,
    load_regulations,
)

from src.preprocessing import (
    declaration_to_text,
    regulation_to_text,
)

from src.bm25 import BM25Retriever
from src.embeddings import EmbeddingRetriever
from src.retrieval import (
    reciprocal_rank_fusion,
)
from src.retrieval import retrieve_candidates
from src.llm_validation import make_llm_prompt, validate_llm_answer, save_llm_answer, load_llm_answers
from src.reranker import Reranker
from src.metrics import precision_at_10, ndcg_at_10

Загрузка датасетов

In [6]:
declarations = load_declarations(
    "./data/declarations (4).jsonl"
)

regulations = load_regulations(
    "./data/regulations (4).jsonl"
)
declarations.head()

,declaration_id,G011,G221,G31_1,G32,G34,G42,GD0,GD00,ND,desc_extention,has_acceptance_docs
0,6091ab306df3425437ab3113,ИМ,,СИСТЕМА ХРАНЕНИЯ ДАННЫХ DATA STORAGE SYSTEM МО...,140,,,10,None,None,:,0
1,609190fc6df3425437aef1af,ИМ,,"АППАРАТУРА ПЕРЕДАЮЩАЯ, ВКЛЮЧАЮЩАЯ В СВОЙ СОСТА...",4,,,10,None,None,:,0
2,60926ff56df3425437dac7c5,ИМ,,"НАСОСЫ МОЛЕКУЛЯРНЫЕ (ВАКУУМНЫЕ), ПРОМЫШЛЕННЫЕ,...",86,,,10,None,None,:,0
3,60941fc06df34254375e2f72,ИМ,,"НАСОСЫ МОЛЕКУЛЯРНЫЕ (ВАКУУМНЫЕ) ПРОМЫШЛЕННЫЕ, ...",90,,,10,None,None,:,0
4,60933b696df34254372e4337,ИМ,,5.2 ИМУЩЕСТВО ПО ПЕРЕЧНЮ №1XXXXX7: ПОЗ. ПО ЛИЦ...,29,,,10,None,None,:,0


In [7]:
regulations.head()

,regulation_id,decree_number,npa,source
0,NPA0001,1661,"Раздел 1, 6.1.4.1.4. зеркала, специально разра...",NaN
1,NPA0002,1661,"Раздел 1, 1.3.12.2. предварительно обогащенный...",NaN
2,NPA0003,1661,"Раздел 1, 8.1.2.1.3. системы, оборудование и к...",NaN
3,NPA0004,36,Угловые измерительные приборы с отклонением уг...,NaN
4,NPA0005,1082,"Дистилляционные или абсорбционные колонны, кот...",NaN


## Анализ данных

In [8]:
print("declaration shape:", declarations.shape)
print("regulation shape:", regulations.shape)

# G32
print("\nG32:")
print("  unique:", declarations["G32"].nunique())
print("  values:", sorted(declarations["G32"].unique())[:10], "...")

# source
print("\nsource:")
print("  non-null:", regulations["source"].notna().sum())
print("  unique non-null:", regulations["source"].dropna().unique())

# Пустые и константные колонки
empty_or_constant = [
    col for col in declarations.columns
    if declarations[col].isna().all()
    or declarations[col].nunique(dropna=False) == 1
]

print("\nПустые или константные колонки:", empty_or_constant)

# Удаляем поля, которые не несут информации для сопоставления
declarations = declarations.drop(columns=empty_or_constant + ["G32"])
regulations = regulations.drop(columns=["source"])

print("\nОставшиеся колонки деклараций:", declarations.columns.tolist())
print("Оставшиеся колонки НПА:", regulations.columns.tolist())

declaration shape: (151, 12)
regulation shape: (562, 4)

G32:
  unique: 151
  values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] ...

source:
  non-null: 19
  unique non-null: ['annotation']

Пустые или константные колонки: ['G011', 'G221', 'G34', 'G42', 'GD0', 'GD00', 'ND', 'desc_extention', 'has_acceptance_docs']

Оставшиеся колонки деклараций: ['declaration_id', 'G31_1']
Оставшиеся колонки НПА: ['regulation_id', 'decree_number', 'npa']


В декларациях поле `G32` содержит 151 уникальное значение для 151 декларации — значения представляют собой числа от 1 до 151 и не повторяются. Поэтому оно не добавляет информации для сопоставления декларации с НПА и исключается из дальнейшего анализа.

Остальные поля деклараций, содержащие только пропуски или одно постоянное значение, также не несут информации для ранжирования и исключаются.

В данных НПА поле `source` заполнено только в 19 из 562 записей, причём единственное встречающееся значение — `annotation`. Поэтому оно также исключается как практически неинформативное для задачи сопоставления.

В результате для дальнейшего анализа используются текстовое описание товара `G31_1` в декларации и текст НПА `npa`. Поля `declaration_id` и `regulation_id` сохраняются как идентификаторы, а `decree_number` — как дополнительная метаинформация о НПА.


In [9]:
print(regulations["decree_number"].nunique())
print(regulations["decree_number"].value_counts().head(20))

7
decree_number
1661      350
202        64
36         58
1005       50
1083       20
1082       12
КЕЭК30      8
Name: count, dtype: int64


`decree_number` содержит 7 различных значений и рассматривается как метаданные НПА, но не используется в текстовом сопоставлении.

In [10]:
# Длины текстов

declarations["text_len"] = declarations["G31_1"].fillna("").str.len()
declarations["word_len"] = declarations["G31_1"].fillna("").str.split().str.len()

regulations["text_len"] = regulations["npa"].fillna("").str.len()
regulations["word_len"] = regulations["npa"].fillna("").str.split().str.len()

print("Длина описаний деклараций:")
print(declarations["text_len"].describe(percentiles=[.5, .75, .9, .95, .99]))

print("\nКоличество слов в декларациях:")
print(declarations["word_len"].describe(percentiles=[.5, .75, .9, .95, .99]))

print("\nДлина НПА:")
print(regulations["text_len"].describe(percentiles=[.5, .75, .9, .95, .99]))

print("\nКоличество слов в НПА:")
print(regulations["word_len"].describe(percentiles=[.5, .75, .9, .95, .99]))


# Пропуски и дубликаты

print("\nПустые тексты:")
print("Декларации:", (declarations["G31_1"].fillna("").str.strip() == "").sum())
print("НПА:", (regulations["npa"].fillna("").str.strip() == "").sum())

print("\nДубликаты текстов:")
print("Декларации:", declarations["G31_1"].duplicated().sum())
print("НПА:", regulations["npa"].duplicated().sum())

print("\nДубликаты ID:")

print("declaration_id:" , declarations["declaration_id"].duplicated().sum())
print("regulation_id:" , regulations["regulation_id"].duplicated().sum())

Длина описаний деклараций:
count     151.000000
mean      431.695364
std       358.399832
min        48.000000
50%       311.000000
75%       555.500000
90%       906.000000
95%      1210.000000
99%      1634.500000
max      2131.000000
Name: text_len, dtype: float64

Количество слов в декларациях:
count    151.000000
mean      53.350993
std       45.788528
min        5.000000
50%       39.000000
75%       67.000000
90%      116.000000
95%      145.000000
99%      201.000000
max      280.000000
Name: word_len, dtype: float64

Длина НПА:
count     562.000000
mean      362.005338
std       334.592659
min        17.000000
50%       259.500000
75%       431.000000
90%       760.100000
95%       939.750000
99%      1549.110000
max      4000.000000
Name: text_len, dtype: float64

Количество слов в НПА:
count    562.000000
mean      45.912811
std       44.043220
min        2.000000
50%       32.000000
75%       53.000000
90%       97.900000
95%      128.000000
99%      198.290000
max      492

Тексты деклараций и НПА не содержат пропусков. Длина варьируется от коротких фрагментов до достаточно длинных описаний (до 280 слов для деклараций и до 492 слов для НПА). Тексты имеют уникальные ID, при этом встречаются отдельные одинаковые тексты, которые сохраняются как разные записи.

In [11]:
declarations["text"] = declarations.apply(
    declaration_to_text,
    axis=1,
)

regulations["text"] = regulations.apply(
    regulation_to_text,
    axis=1,
)

Используется минимальная нормализация:

- Unicode NFKC;
- приведение к нижнему регистру;
- нормализация пробелов.

Пунктуация, числа, единицы измерения и технические обозначения
сохраняются, поскольку они могут быть существенными для сопоставления
товара с НПА.

Лемматизация и удаление стоп-слов не применяются.

## Лексический baseline — BM25

В качестве простого baseline используется BM25. Он позволяет проверить,
насколько релевантные НПА можно находить только по лексическому
пересечению текста.

Для качественной проверки рассматриваются несколько деклараций и
первые пять результатов.

In [12]:
bm25 = BM25Retriever(
    regulations["text"].tolist()
)

In [13]:
for i in [0, 10, 20]:
    query = declarations.iloc[i]["text"]

    results = bm25.retrieve(query, top_k=5)

    print("=" * 80)
    print("DECLARATION:", declarations.iloc[i]["declaration_id"])
    print(query[:500])

    for rank, (idx, score) in enumerate(results, 1):
        print(f"\n{rank}. {regulations.iloc[idx]['regulation_id']} | score={score:.3f}")
        print(regulations.iloc[idx]["text"][:300])

DECLARATION: 6091ab306df3425437ab3113
система хранения данных data storage system мод.e03t (scv3020) память: 15 твердотельных накопителей по 1.92 tb, sas, 12 гбит/с+ 15 т вердотельных накопителей по 2.4 tb, sas, 12 гбит/с плата ввода-вывода, fc 16 гбит/с, 4 порта, pci-e, полновысотная

1. NPA0190 | score=23.470
раздел 1, 3.1.2.1.6. устройства записи цифровых данных, удовлетворяющие всем следующим условиям: а) обладающие устойчивой пропускной способностью диска или твердотельной памяти более 6,4 гбит/с; и б) использующие процессор, выполняющий анализ параметров радиочастотного сигнала одновременно с его зап

2. NPA0013 | score=18.850
раздел 1, 3.1.2.7. электронные сборки, модули или оборудование, предназначенные для выполнения всего следующего: а) аналого-цифровых преобразований, имеющих любую из следующих характеристик: разрешающую способность 8 бит или более, но менее 10 бит с частотой выборки более 1,3 млрд. выборок в секунду

3. NPA0364 | score=12.401
раздел 1, 1.3.2.1.1. алюминиды 

BM25 способен находить НПА с совпадающими техническими терминами, однако
при отсутствии буквального совпадения слов в описании товара и НПА
релевантность может быть потеряна.

Поэтому дополнительно проверяется семантический retrieval.

## Семантический retrieval — BGE-M3

Для поиска семантически близких НПА используется
`BAAI/bge-m3`.

Модель применяется для получения эмбеддингов описаний НПА и деклараций.
Поскольку в датасете всего 562 НПА, все документы можно заранее
закодировать, а для каждой декларации выполнять быстрый поиск по
всему корпусу.

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_retriever = EmbeddingRetriever(
    model_path="./models/bge-m3",
    device=device,
    batch_size=32,
)
embedding_retriever.fit(
    regulations["text"].tolist()
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

In [17]:
for i in [0, 10, 20]:
    query = declarations.iloc[i]["text"]

    results = embedding_retriever.retrieve(query, top_k=5)

    print("=" * 80)
    print("DECLARATION:", declarations.iloc[i]["declaration_id"])
    print(query[:500])

    for rank, (idx, score) in enumerate(results, 1):
        print(
            f"\n{rank}. "
            f"{regulations.iloc[idx]['regulation_id']} "
            f"| score={score:.3f}"
        )
        print(regulations.iloc[idx]["text"][:300])

DECLARATION: 6091ab306df3425437ab3113
система хранения данных data storage system мод.e03t (scv3020) память: 15 твердотельных накопителей по 1.92 tb, sas, 12 гбит/с+ 15 т вердотельных накопителей по 2.4 tb, sas, 12 гбит/с плата ввода-вывода, fc 16 гбит/с, 4 порта, pci-e, полновысотная

1. NPA0190 | score=0.593
раздел 1, 3.1.2.1.6. устройства записи цифровых данных, удовлетворяющие всем следующим условиям: а) обладающие устойчивой пропускной способностью диска или твердотельной памяти более 6,4 гбит/с; и б) использующие процессор, выполняющий анализ параметров радиочастотного сигнала одновременно с его зап

2. NPA0398 | score=0.540
раздел 1, 3.1.1.2.8. мощные свч-модули, содержащие, по крайней мере, вакуумное электронное устройство бегущей волны, монолитную микроволновую интегральную схему и встроенный электронный стабилизатор напряжения, имеющие все следующие характеристики: а) время включения от выключенного состояния до пол

3. NPA0559 | score=0.516
схемы электронные интегральные, за

BGE-M3 позволяет учитывать семантическую близость даже в случаях,
когда описание товара и НПА используют разные формулировки.

Таким образом, BM25 и BGE-M3 дают два разных сигнала:
лексический и семантический.

## Диагностическая разметка

Поскольку исходный датасет не содержит ground truth, для сравнения
методов используется небольшая диагностическая выборка из 20 деклараций.

Для кандидатов из объединённого пула BM25 + BGE-M3 была получена
слабая разметка с помощью LLM (ChatGPT).

Разметка бинарная:

- `1` — между товаром и НПА видна содержательная связь;
- `0` — содержательная связь по предоставленным текстам не видна.

Разметка используется только для сравнения методов и не рассматривается
как истинная эталонная разметка.

In [169]:
i = 19

declaration = declarations.iloc[i]

candidates_with_scores = retrieve_candidates(
    declaration["text"],
    bm25,
    embedding_retriever,
    candidate_k=100,
    bm25_k=100,
    embedding_k=100,
)
candidates = [
    {
        "regulation_id": regulations.iloc[idx]["regulation_id"],
        "npa": regulations.iloc[idx]["text"],
    }
    for idx, _ in candidates_with_scores['rrf']
]

In [ ]:
prompt = make_llm_prompt(
    declaration_id=declaration['declaration_id'],
    declaration_text=declaration['text'],
    candidates=candidates,
)

print(prompt)

In [ ]:
answer = """{
"declaration_id": "6091f1746df34254370f8e56",
"results": [
{"regulation_id": "NPA0093", "relevance": 0},
...
]
}
"""

answer = validate_llm_answer(
    answer,
    [
        regulations.iloc[idx]["regulation_id"]
        for idx, _ in candidates_with_scores["rrf"]
    ]
)

In [ ]:
save_llm_answer(
    answer,
    "data/llm_answers.jsonl"
)

## Сравнение методов

Для выбора метода были рассмотрены пять вариантов:

- BM25;
- BGE-M3;
- BM25 + BGE-M3 через RRF;
- BGE-M3 + reranker;
- RRF + reranker.

На диагностической выборке из 20 деклараций для каждого варианта
рассчитывались Precision@10 и NDCG@10 относительно weak labels,
полученных для кандидатов.

Полученные результаты используются именно для сравнительного анализа
методов. При этом weak labels не являются эталонной разметкой, поэтому
метрики характеризуют поведение методов на данной диагностической
выборке и не рассматриваются как абсолютная оценка качества.

In [ ]:
reranker = Reranker(
    model_path="./models/bge-reranker-v2-m3",
    device=device,
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
llm_answers = load_llm_answers("data/llm_answers.jsonl")

results = []

for i in range(20):

    declaration = declarations.iloc[i]
    declaration_id = str(declaration["declaration_id"])
    query = declaration["text"]

    if declaration_id not in llm_answers:
        print(f"Нет разметки для {declaration_id}")
        continue

    rankings = retrieve_candidates(
        query,
        bm25,
        embedding_retriever,
        candidate_k=100,
        bm25_k=100,
        embedding_k=100,
    )

    relevance = llm_answers[declaration_id]

    # -------------------------
    # RRF -> Reranker
    # -------------------------

    fused = rankings["rrf"]

    rrf_candidate_indices = [
        idx for idx, _ in fused
    ]

    rrf_candidates = [
        (
            idx,
            regulations.iloc[idx]["text"],
        )
        for idx in rrf_candidate_indices
    ]

    rrf_reranked = reranker.rerank(
        query,
        rrf_candidates,
        top_k=10,
    )

    rankings["rrf_reranker"] = rrf_reranked

    # -------------------------
    # BGE -> Reranker
    # -------------------------

    bge = rankings["bge"]

    bge_candidate_indices = [
        idx for idx, _ in bge
    ]

    bge_candidates = [
        (
            idx,
            regulations.iloc[idx]["text"],
        )
        for idx in bge_candidate_indices
    ]

    bge_reranked = reranker.rerank(
        query,
        bge_candidates,
        top_k=10,
    )

    rankings["bge_reranker"] = bge_reranked

    # -------------------------
    # Metrics
    # -------------------------

    for method, ranking in rankings.items():

        ranking_ids = [
            (
                regulations.iloc[idx]["regulation_id"],
                score,
            )
            for idx, score in ranking
        ]

        results.append({
            "declaration_id": declaration_id,
            "method": method,
            "precision@10": precision_at_10(
                ranking_ids,
                relevance,
            ),
            "ndcg@10": ndcg_at_10(
                ranking_ids,
                relevance,
            ),
            "relevant_in_top10": sum(
                relevance.get(regulation_id, 0) == 1
                for regulation_id, _ in ranking_ids[:10]
            ),
        })


metrics_df = pd.DataFrame(results)

In [33]:
summary = (
    metrics_df
    .groupby("method")
    [["precision@10", "ndcg@10", "relevant_in_top10"]]
    .mean()
    .sort_values("ndcg@10", ascending=False)
)

summary

,precision@10,ndcg@10,relevant_in_top10
method,,,
bge,0.175,0.529643,1.75
rrf,0.100,0.440156,1.00
rrf_reranker,0.160,0.432560,1.60
bge_reranker,0.150,0.405700,1.50
bm25,0.080,0.353914,0.80


По результатам диагностического эксперимента BGE-M3 показал наибольшие
значения Precision@10 и NDCG@10 среди рассмотренных вариантов.

Добавление BM25 через RRF и последующее переранжирование не улучшило
результат относительно непосредственного использования BGE-M3 на этой
выборке.

Поэтому в финальный pipeline выбран BGE-M3 без дополнительных этапов
fusion и reranking.

## Итоговый запуск

In [35]:
!python run.py --out ./out

Device: cuda
Declarations: 151
Regulations: 562
Loading BGE-M3...
Loading weights: 100% 391/391 [00:00<00:00, 27064.04it/s]
Encoding regulations...
Batches: 100% 18/18 [00:15<00:00,  1.18it/s]
Processed 10/151
Processed 20/151
Processed 30/151
Processed 40/151
Processed 50/151
Processed 60/151
Processed 70/151
Processed 80/151
Processed 90/151
Processed 100/151
Processed 110/151
Processed 120/151
Processed 130/151
Processed 140/151
Processed 150/151

Done!
Saved to: out/predictions.csv
Rows: 1510
